In [7]:
import numpy as np
from prettytable import PrettyTable

In [8]:
friends = ["Alice", "Bob", "Carol", "David"]

# Rows: Who paid
# Columns: Who the expense was for
expense_matrix = np.zeros((len(friends), len(friends)))

In [16]:
expense_matrix


array([[416.66666667, 416.66666667, 416.66666667,   0.        ],
       [  0.        , 400.        , 400.        ,   0.        ],
       [446.25      , 446.25      , 446.25      , 446.25      ],
       [  0.        ,   0.        ,   0.        ,   0.        ]])

In [9]:
def add_expense(payer, beneficiaries, amount):
    payer_idx = friends.index(payer)

    # Calculate equal share
    share_per_person = amount / len(beneficiaries)

    # Update the expense matrix
    for beneficiary in beneficiaries:
        beneficiary_idx = friends.index(beneficiary)
        expense_matrix[payer_idx][beneficiary_idx] += share_per_person

In [10]:
def calculate_settlements():
    # Total amount each person paid (Row Sum)
    total_paid = np.sum(expense_matrix, axis=1)

    # Total amount each person owes (Column Sum)
    total_owed = np.sum(expense_matrix, axis=0)

    # Positive = Should Receive
    # Negative = Owes Money
    net_balance = total_paid - total_owed

    return net_balance

In [11]:
def display_settlements():
    settlements = calculate_settlements()

    table = PrettyTable()
    table.field_names = ["Friend", "Settlement"]

    for i, friend in enumerate(friends):

        if settlements[i] > 0:
            table.add_row([friend, f"Should Receive ₹{settlements[i]:.2f}"])

        elif settlements[i] < 0:
            table.add_row([friend, f"Owes ₹{-settlements[i]:.2f}"])

        else:
            table.add_row([friend, "Is Settled"])

    print("\nFinal Settlements:")
    print(table)

In [12]:
def suggest_payments():

    settlements = calculate_settlements()

    # People who should receive money
    creditors = [
        (friends[i], amt)
        for i, amt in enumerate(settlements)
        if amt > 0
    ]

    # People who should pay money
    debtors = [
        (friends[i], -amt)
        for i, amt in enumerate(settlements)
        if amt < 0
    ]

    transactions = []

    # Match debtors with creditors
    while debtors and creditors:

        debtor, debt_amount = debtors.pop(0)
        creditor, credit_amount = creditors.pop(0)

        payment = min(debt_amount, credit_amount)

        transactions.append((debtor, creditor, payment))

        # Update balances
        debt_amount -= payment
        credit_amount -= payment

        # Put back remaining balances
        if debt_amount > 0:
            debtors.insert(0, (debtor, debt_amount))

        if credit_amount > 0:
            creditors.insert(0, (creditor, credit_amount))

    print("\nSuggested Transactions:")

    if transactions:
        for debtor, creditor, amount in transactions:
            print(f"{debtor} should pay ₹{amount:.2f} to {creditor}")
    else:
        print("No transactions needed. Everyone is settled.")

In [13]:
# Alice paid ₹1250 for Alice, Bob and Carol
add_expense("Alice", ["Alice", "Bob", "Carol"], 1250)

# Bob paid ₹800 for Bob and Carol
add_expense("Bob", ["Bob", "Carol"], 800)

# Carol paid ₹1785 for everyone
add_expense("Carol", ["Alice", "Bob", "Carol", "David"], 1785)

In [14]:
display_settlements()


Final Settlements:
+--------+------------------------+
| Friend |       Settlement       |
+--------+------------------------+
| Alice  | Should Receive ₹387.08 |
|  Bob   |      Owes ₹462.92      |
| Carol  | Should Receive ₹522.08 |
| David  |      Owes ₹446.25      |
+--------+------------------------+


In [15]:
suggest_payments()


Suggested Transactions:
Bob should pay ₹387.08 to Alice
Bob should pay ₹75.83 to Carol
David should pay ₹446.25 to Carol
